<a href="https://colab.research.google.com/github/claimar/TransferLearningforAnimalSounds/blob/main/modeling_inputs/notebooks/remote_sensing_data/Remote_Sensing_Covariates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction
**Author:** Claudia I. Martinez Garcia · **Project:** Heat-Wave Vulnerability Mapping (IRIU / Mitacs-IVADO)

This notebook aims to compute environmental exposure variables from satellite imagery using Google Earth Engine (GEE) for the 484 census tracts of Montreal between 2000 and 2025. These variables, derived through remote sensing, characterize the physical environment and are commonly used in climate and environmental health studies.

The main indicators computed in this notebook are the Normalized Difference Vegetation Index (NDVI) and Green Cover Fraction, which serve as proxies for urban greenness and vegetation cover. These variables are important for heatwave vulnerability modeling because vegetation can reduce local temperatures through shading and evapotranspiration, helping mitigate the urban heat island effect.

The resulting dataset will be integrated with socio-demographic and hazard variables to support the development of heatwave vulnerability models at the census-tract level.


# Notebook Requirements

This notebook is currently configured to use Google Earth Engine (GEE) assets together with local project files stored in the `IRIU_VF` directory.

The required local files are available in the repository under:

`modeling_inputs/datasets/notebook_dependencies/remote_sensing_covariates/`

If you store these files in a different location, update the corresponding file paths and Google Earth Engine asset paths in the notebook before execution.

## Required Local Files

### Montréal Boundary

- `montreal.geojson`

## Required Google Earth Engine Assets

- Montréal 2021 Census Tracts (`TRACTS_ASSET`)

> The census tract asset should correspond to the 2021 Montréal census tracts and contain the attributes required by the notebook (e.g., `CTUID`, `CTNAME`, and `DGUID`).

## Required Google Earth Engine Authentication

Before running the notebook, authenticate and initialize Google Earth Engine.

```python
import ee

ee.Authenticate()
ee.Initialize(project='your-google-cloud-project')
```

# 1. NDVI & Green Cover — Montreal Census Tracts (Landsat, 2000–2025)

This section computes a historical **NDVI** and **green-cover** series for the
City of Montreal's **484 census tracts**, from Landsat surface reflectance in
Google Earth Engine, and exports per-tract / per-year tables (and optional
rasters + PNGs) for merging with health and vulnerability data.

Project Overview

**Goal.** A reproducible, per-tract, per-year green-cover covariate for Montreal.

**Pipeline**
1. **Tracts** — load the 484 City-of-Montreal census tracts (pre-clipped asset).
2. **Imagery** — Landsat Collection 2 Level-2 surface reflectance, summer
   (Jun 1 – Sep 30) composites per year.
   - 2000–2011 → Landsat 5 TM · 2012 → Landsat 7 ETM+ · 2013+ → Landsat 8/9 OLI
3. **Masking** — cloud, dilated cloud, cloud shadow, snow (QA_PIXEL).
4. **Scaling** — Collection-2 scale/offset → true reflectance (essential; the
   −0.2 offset does **not** cancel in NDVI).
5. **NDVI** — `normalizedDifference(NIR, Red)` (safe division).
6. **Green cover** — `NDVI > threshold` → 0/1 mask → per-tract mean = fraction.
7. **Validate, visualize, export** (CSV always; rasters/PNG optional).





## 1.2 Configuration


In [ ]:
# ----------------------------- EDIT ME -------------------------------
# Earth Engine project + input assets
EE_PROJECT    = "montreal-green-cover"                                    # TODO confirm
TRACTS_ASSET  = "projects/montreal-green-cover/assets/montreal_2021_cts_484"  # 484 CTs
VILLE_ASSET   = "projects/montreal-green-cover/assets/ville_montreal"     # city boundary

# Output (Google Drive)
DRIVE_OUTPUT_FOLDER = "IRIU_VF"          # TODO confirm Drive folder name

# Time window
START_YEAR, END_YEAR = 2000, 2025
SUMMER_START, SUMMER_END = (6, 1), (9, 30)   # (month, day)

# Band mapping per sensor family (Collection 2 L2)
#   Landsat 5/7 (TM/ETM+): NIR=SR_B4, RED=SR_B3
#   Landsat 8/9 (OLI):     NIR=SR_B5, RED=SR_B4
BANDS_L57 = {"NIR": "SR_B4", "RED": "SR_B3"}
BANDS_L89 = {"NIR": "SR_B5", "RED": "SR_B4"}

# NDVI / green cover
NDVI_THRESHOLD = 0.3        # vegetation cutoff (sensitivity-tested below)
MASK_WATER     = True       # True = green share of LAND; False = of total tract area

# Reduction / compute
SCALE_M    = 30             # native Landsat optical resolution (do NOT use 60)
TILE_SCALE = 4              # raise to 8/16 if you hit memory/timeout errors

# Collection-2 surface-reflectance scaling (fixed by USGS; do not change)
SR_SCALE, SR_OFFSET = 0.0000275, -0.2

# QA_PIXEL bits to mask (Collection 2)
QA_BITS = {"dilated_cloud": 1, "cloud": 3, "cloud_shadow": 4, "snow": 5}

# Raster / PNG export options
EXPORT_RASTERS = False      # True also exports NDVI + green rasters to Drive
EXPORT_PNG     = True       # save quick-look PNGs locally for download
EXPORT_CRS     = "EPSG:32618"   # UTM 18N (Montreal); used only for raster export
PNG_YEAR       = 2015       # year used for the visualization/PNG section

# Validation
STABLE_CTUIDS  = []         # TODO: list of CTUID strings for large parks/cemeteries
VALIDATION_YEARS = [2002, 2012, 2015, 2024]

# Export file names (CSV always written)
EXPORT_NAMES = {
    "p1": "ville_montreal_ct_ndvi_green_2000_2011",
    "p2": "ville_montreal_ct_ndvi_green_2012",
    "p3": "ville_montreal_ct_ndvi_green_2013_2025",
}
# ---------------------------------------------------------------------

# Re-initialize EE with the real project id from this cell
import ee
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=EE_PROJECT)

print("Config loaded. EE project:", EE_PROJECT)
print(f"Years {START_YEAR}-{END_YEAR} | NDVI>{NDVI_THRESHOLD} | "
      f"scale={SCALE_M} m | mask_water={MASK_WATER}")


To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=2ZOMNh7-hzk287A0dlLtBaD8cJ7ajgpF7r3yuMPPmbQ&tc=1PPIjK1S1tK9eAV3nnJ9zpHEsrJ49xWr0dUp3wAEcRY&cc=zVDt2fMFuD9P0XEICSrGKqs_Wr4RXdaY1cJgvMq8oKw

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPx6wEZus8fnyku83nQeClrV7Yd6KcUZLANqpJOnyY1Gdtz9Y86UfHE

Successfully saved authorization token.
Config loaded. EE project: montreal-green-cover
Years 2000-2025 | NDVI>0.3 | scale=30 m | mask_water=True


# 1.3 Data Loading & Validation

Load the tracts, confirm the count, and prepare the optional water mask. We
validate **existence, geometry type, count, and CRS assumptions** before compute.

In [ ]:
# Load census tracts (already clipped to City of Montreal = 484 CTs)
tracts = ee.FeatureCollection(TRACTS_ASSET)

n_tracts = tracts.size().getInfo()
print("Tracts loaded:", n_tracts)

# --- validation: count + property schema + a sample geometry ---
assert n_tracts > 0, "Tract collection is empty — check TRACTS_ASSET."
if n_tracts != 484:
    print(f"WARNING: expected 484 tracts, got {n_tracts}. "
          "Confirm the City-of-Montreal filter (see TODO).")

props = tracts.first().propertyNames().getInfo()
print("Tract properties:", props)
for key in ("CTUID", "DGUID"):
    if key not in props:
        print(f"WARNING: '{key}' not found in tract properties — exports/merge may break.")

# EE FeatureCollections are geographic (EPSG:4326) by convention; reductions are
# done in the image projection at SCALE_M. EXPORT_CRS is applied only to rasters.
print("CRS note: reductions use the Landsat image projection at",
      SCALE_M, "m; raster exports reproject to", EXPORT_CRS)

Tracts loaded: 484
Tract properties: ['DGUID', 'PRUID', 'CTNAME', 'inside_ville', 'CTUID', 'LANDAREA', 'system:index']
CRS note: reductions use the Landsat image projection at 30 m; raster exports reproject to EPSG:32618


In [ ]:
# Optional stable water mask (JRC Global Surface Water) to keep water out of the
# GREEN numerator. Water already isn't 'green', but this also enables land-only %.
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
water_mask = jrc.gt(50).unmask(0)     # pixels with >50% water occurrence
not_water  = water_mask.Not()         # 1 = land, 0 = water
print("Water mask ready (used only if MASK_WATER = True):", bool(MASK_WATER))

Water mask ready (used only if MASK_WATER = True): True


## 1.4 NDVI Calculation

`NDVI = (NIR − Red) / (NIR + Red)` via `normalizedDifference()` (handles the
`NIR+Red = 0` division-by-zero case). Surface reflectance is **scaled first**
(scale ×0.0000275, offset −0.2); the offset does not cancel in the ratio, so
skipping it biases NDVI and the threshold.

In [ ]:
def mask_landsat_sr(image):
    """Mask cloud, dilated cloud, cloud shadow, and snow using QA_PIXEL."""
    qa = image.select("QA_PIXEL")
    mask = qa.bitwiseAnd(1 << QA_BITS["dilated_cloud"]).eq(0)
    for b in ("cloud", "cloud_shadow", "snow"):
        mask = mask.And(qa.bitwiseAnd(1 << QA_BITS[b]).eq(0))
    return image.updateMask(mask)


def _add_ndvi_green(image, nir_band, red_band):
    """Scale SR, compute NDVI (safe division), add binary green-cover band."""
    scaled = (image.select([nir_band, red_band])
              .multiply(SR_SCALE).add(SR_OFFSET)
              .rename(["NIR", "RED"]))
    ndvi  = scaled.normalizedDifference(["NIR", "RED"]).rename("NDVI")
    green = ndvi.gt(NDVI_THRESHOLD).rename("green_cover")
    return image.addBands([ndvi, green])


def prep_l57(image):
    image = mask_landsat_sr(image)
    return _add_ndvi_green(image, BANDS_L57["NIR"], BANDS_L57["RED"])


def prep_l89(image):
    image = mask_landsat_sr(image)
    return _add_ndvi_green(image, BANDS_L89["NIR"], BANDS_L89["RED"])

print("NDVI / green-cover functions defined.")

NDVI / green-cover functions defined.


In [ ]:
def get_collection(year):
    """Return the masked, NDVI-tagged collection for one summer, sensor-selected."""
    year  = ee.Number(year)
    start = ee.Date.fromYMD(year, SUMMER_START[0], SUMMER_START[1])
    end   = ee.Date.fromYMD(year, SUMMER_END[0],   SUMMER_END[1])

    def coll(cid, prep):
        return (ee.ImageCollection(cid)
                .filterDate(start, end)
                .filterBounds(tracts)
                .map(prep))

    l5 = coll("LANDSAT/LT05/C02/T1_L2", prep_l57)
    l7 = coll("LANDSAT/LE07/C02/T1_L2", prep_l57)
    l8 = coll("LANDSAT/LC08/C02/T1_L2", prep_l89)
    l9 = coll("LANDSAT/LC09/C02/T1_L2", prep_l89)

    return ee.ImageCollection(
        ee.Algorithms.If(
            year.lte(2011), l5,
            ee.Algorithms.If(year.eq(2012), l7, l8.merge(l9))
        )
    )

print("Sensor-selection function defined (L5 <=2011, L7 =2012, L8/9 >=2013).")


Sensor-selection function defined (L5 <=2011, L7 =2012, L8/9 >=2013).


## 1.5 Green Cover Calculation

Per year: median composite → optional water mask → per-tract mean of the 0/1
vegetation band = **green-cover fraction** (0–1). We also carry `n_scenes` so
thin years are detectable rather than silently NaN. `green_cover_percent`
(0–100) and an approximate vegetated **area** (km², tract area × fraction) are added.


In [ ]:
def compute_year(year):
    """Per-tract NDVI mean + green-cover fraction for one year, with metadata."""
    year = ee.Number(year)
    coll = get_collection(year).select(["NDVI", "green_cover"])
    n_scenes = coll.size()

    # median of 0/1 green band = per-pixel majority vegetation; then spatial mean
    composite = coll.median()
    if MASK_WATER:
        composite = composite.updateMask(not_water)

    result = composite.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=SCALE_M,
        tileScale=TILE_SCALE,
    )

    def tag(f):
        gp = ee.Number(f.get("green_cover"))            # fraction 0..1 (may be null)
        # tract area in km^2 for an approximate vegetated-area figure
        area_km2 = f.geometry().area(1).divide(1e6)
        return f.set({
            "year": year,
            "n_scenes": n_scenes,
            "green_cover_percent": ee.Algorithms.If(gp, gp.multiply(100), None),
            "veg_area_km2": ee.Algorithms.If(gp, gp.multiply(area_km2), None),
            "tract_area_km2": area_km2,
            "satellite": ee.Algorithms.If(
                year.lte(2011), "Landsat 5",
                ee.Algorithms.If(year.eq(2012), "Landsat 7", "Landsat 8/9")),
        })

    return result.map(tag)

print("compute_year() defined.")


compute_year() defined.


## 1.6 Validation Checks

Lightweight `getInfo()` pulls (no exports). Confirm: NDVI within [−1, 1] with
sensible summer means; `green_cover` strictly in [0, 1]; `n_scenes` not too low;
and that the 0.2 / 0.3 / 0.4 threshold sweep moves green cover smoothly.

In [ ]:
def summarize_year(y):
    """Print NDVI min/mean/max and scene count for one year."""
    fc = ee.FeatureCollection(ee.List([y]).map(compute_year)).flatten()
    stats = fc.reduceColumns(
        reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
        selectors=["NDVI"],
    ).getInfo()
    n = fc.aggregate_array("n_scenes").get(0).getInfo()
    print(f"Year {y}: NDVI {stats} | scenes={n}")
    if n is not None and n < 4:
        print(f"   WARNING: only {n} scene(s) for {y} — composite may be unreliable.")

for y in VALIDATION_YEARS:
    summarize_year(y)

Year 2002: NDVI {'max': 0.7413632455036067, 'mean': 0.36932214167903626, 'min': 0.1368028106766709} | scenes=15
Year 2012: NDVI {'max': 0.7750766938007941, 'mean': 0.37813832392469726, 'min': 0.10474259698974918} | scenes=19
Year 2015: NDVI {'max': 0.7662464494880273, 'mean': 0.4011281194945428, 'min': 0.13429660946496966} | scenes=21
Year 2024: NDVI {'max': 0.7650842124848617, 'mean': 0.40293999266135244, 'min': 0.1555239179152064} | scenes=37


In [ ]:
# green_cover must be a fraction in [0, 1]
gc = ee.FeatureCollection(ee.List([PNG_YEAR]).map(compute_year)).flatten()
gc_stats = gc.reduceColumns(ee.Reducer.minMax(), ["green_cover"]).getInfo()
print(f"{PNG_YEAR} green_cover min/max (expect 0..1):", gc_stats)

mn, mx = gc_stats.get("min"), gc_stats.get("max")
assert (mn is None) or (mn >= -1e-9 and mx <= 1 + 1e-9), \
    "green_cover out of [0,1] — check masking/threshold."
print("green_cover range OK.")



2015 green_cover min/max (expect 0..1): {'max': 0.9972674836494538, 'min': 0.01826968654040352}
green_cover range OK.


In [ ]:
# Threshold sensitivity: re-run one year at 0.2 / 0.3 / 0.4, compare mean green cover.
def green_at(y, thr):
    comp = get_collection(y).select("NDVI").median()
    green = comp.gt(thr).rename("g")
    if MASK_WATER:
        green = green.updateMask(not_water)
    stat = green.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=tracts.geometry(),
        scale=SCALE_M,
        tileScale=TILE_SCALE,
        maxPixels=1e13,
    )
    return stat.get("g").getInfo()   # band is named "g", so this key exists

In [ ]:
print(f"Threshold sensitivity for {PNG_YEAR}:")
for thr in (0.2, 0.3, 0.4):
    val = green_at(PNG_YEAR, thr)
    print(f"NDVI>{thr}: mean green cover = {round(val, 3) if val is not None else None}")

Threshold sensitivity for 2015:
NDVI>0.2: mean green cover = 0.823
NDVI>0.3: mean green cover = 0.705
NDVI>0.4: mean green cover = 0.566


# Interpretation:
#
A sensitivity analysis was performed using NDVI thresholds of 0.20,
 0.30, and 0.40. Estimated city-wide green cover ranged from 56.6%
to 82.3%, indicating that the threshold influences the absolute level
of green cover but not the overall spatial interpretation.
#
The threshold of NDVI > 0.30 was retained because it is widely used
in urban remote-sensing studies and provides a balance between
capturing sparse vegetation and excluding non-vegetated surfaces.
#
Therefore, the green-cover variable is considered reasonably robust,
although alternative thresholds may be explored in future sensitivity
analyses.

## 1.7 Visualization

An interactive `geemap` map of NDVI and the green mask for `PNG_YEAR`, plus a
static matplotlib bar chart of city-wide mean green cover over time (built from
the validation pulls). Interactive maps don't render in a static PDF export but
work live in Colab.


In [ ]:
import geemap

comp = get_collection(PNG_YEAR).select(["NDVI", "green_cover"]).median()
if MASK_WATER:
    comp = comp.updateMask(not_water)

Map = geemap.Map(center=[45.55, -73.62], zoom=10)
Map.addLayer(comp.select("NDVI"),
             {"min": 0, "max": 0.8, "palette": ["white", "yellow", "green"]},
             f"NDVI {PNG_YEAR}")
Map.addLayer(comp.select("green_cover").selfMask(),
             {"palette": ["green"]}, f"Green mask {PNG_YEAR}")
Map.addLayer(tracts.style(color="black", fillColor="00000000", width=1),
             {}, "Census tracts")
Map  # display


Map(center=[45.55, -73.62], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## 1.8 Exports

CSV tables (always) in three blocks, matching the original workflow. Optional
NDVI / green rasters to Drive, and the PNG above for quick download. EE table/raster
exports go to **Drive**; monitor them in the EE *Tasks* tab. Local files can be
pulled with `google.colab.files.downloa

In [ ]:
def export_block(year_list, name):
    fc = ee.FeatureCollection(ee.List(year_list).map(compute_year)).flatten()
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=name,
        folder=DRIVE_OUTPUT_FOLDER,
        fileFormat="CSV",
        selectors=["year", "satellite", "n_scenes", "CTUID", "CTNAME", "DGUID",
                   "NDVI", "green_cover", "green_cover_percent",
                   "veg_area_km2", "tract_area_km2"],
    )
    task.start()
    print(f"Export started -> {name} (Drive/{DRIVE_OUTPUT_FOLDER})")
    return task

# Three blocks, as in the original notebook
t1 = export_block(list(range(2000, 2012)), EXPORT_NAMES["p1"])   # 2000-2011 (L5)
t2 = export_block([2012],                  EXPORT_NAMES["p2"])   # 2012 (L7)
t3 = export_block(list(range(2013, END_YEAR + 1)), EXPORT_NAMES["p3"])  # 2013+ (L8/9)
print("All CSV export tasks submitted. Watch the EE 'Tasks' tab for completion.")


Export started -> ville_montreal_ct_ndvi_green_2000_2011 (Drive/IRIU_VF)
Export started -> ville_montreal_ct_ndvi_green_2012 (Drive/IRIU_VF)
Export started -> ville_montreal_ct_ndvi_green_2013_2025 (Drive/IRIU_VF)
All CSV export tasks submitted. Watch the EE 'Tasks' tab for completion.


---
## 2. Impervious Surface / Built-up (NDBI + Dynamic World)

Adds an **impervious-surface exposure covariate** alongside green cover, using the
same Landsat pipeline so it shares the full 2000–2025 history and the green-cover
output format.

**Three products:**
1. **NDBI continuous** (Landsat) — per-pixel built-up index, full history.
2. **Impervious % per tract** (Landsat, NDBI thresholded) — interpretable fraction, full history.
3. **GHLS" %** (2015+) — use to calibrate the threshold.

**NDBI** = (SWIR1 − NIR) / (SWIR1 + NIR). Bands differ by sensor:
Landsat 5/7 → SWIR1 = `SR_B5`, NIR = `SR_B4`; Landsat 8/9 → SWIR1 = `SR_B6`, NIR = `SR_B5`.

> Reuses `tracts`, `not_water`, `mask_landsat_sr`, `SCALE_M`, `TILE_SCALE`,
> `SR_SCALE/SR_OFFSET`, `SUMMER_*`, `MASK_WATER` from the cells above. Run those first.

In [ ]:
# --- Impervious / NDBI config (edit here) -----------------------------
# SWIR1/NIR band mapping per sensor family (Collection 2 L2)
BANDS_L57_NDBI = {"SWIR1": "SR_B5", "NIR": "SR_B4"}   # Landsat 5/7
BANDS_L89_NDBI = {"SWIR1": "SR_B6", "NIR": "SR_B5"}   # Landsat 8/9

# Built-up cutoff is SET BY THE CALIBRATION CELL (against GHSL), not hard-coded.
NDBI_THRESHOLD = None         # auto-set by the GHSL calibration step below

# NOTE: NO WATER MASK is applied anywhere in the built-up path (per methodology
# decision). Water has low NDBI and GHSL reports ~0 built over water, so water
# pixels do not inflate the built-up fraction; masking is unnecessary and caused
# alignment problems, so it has been removed entirely.

# Export file names for the impervious series (CSV)
EXPORT_NAMES_IMPERV = {
    "p1": "ville_montreal_ct_ndbi_imperv_2000_2011",
    "p2": "ville_montreal_ct_ndbi_imperv_2012",
    "p3": "ville_montreal_ct_ndbi_imperv_2013_2025",
}

print(f"Impervious config loaded | NDBI threshold={NDBI_THRESHOLD} "
      f"(set by GHSL calibration) | NO water mask")

Impervious config loaded | NDBI threshold=None (set by GHSL calibration) | NO water mask


In [ ]:
# --- NDBI / built-up band functions (mirror the NDVI/green design) -----
def _add_ndbi_builtup(image, swir1_band, nir_band):
    scaled = (image.select([swir1_band, nir_band])
              .multiply(SR_SCALE).add(SR_OFFSET)
              .rename(["SWIR1", "NIR"]))

    ndbi = scaled.normalizedDifference(["SWIR1", "NIR"]).rename("NDBI")

    out = image.addBands(ndbi)

    if NDBI_THRESHOLD is not None:
        builtup = ndbi.gt(NDBI_THRESHOLD).rename("builtup")
        out = out.addBands(builtup)

    return out


def prep_l57_ndbi(image):
    image = mask_landsat_sr(image)        # reuse the same QA mask
    return _add_ndbi_builtup(image, BANDS_L57_NDBI["SWIR1"], BANDS_L57_NDBI["NIR"])


def prep_l89_ndbi(image):
    image = mask_landsat_sr(image)
    return _add_ndbi_builtup(image, BANDS_L89_NDBI["SWIR1"], BANDS_L89_NDBI["NIR"])


def get_collection_ndbi(year):
    """Masked, NDBI-tagged Landsat collection for one summer, sensor-selected."""
    year  = ee.Number(year)
    start = ee.Date.fromYMD(year, SUMMER_START[0], SUMMER_START[1])
    end   = ee.Date.fromYMD(year, SUMMER_END[0],   SUMMER_END[1])

    def coll(cid, prep):
        return (ee.ImageCollection(cid)
                .filterDate(start, end)
                .filterBounds(tracts)
                .map(prep))

    l5 = coll("LANDSAT/LT05/C02/T1_L2", prep_l57_ndbi)
    l7 = coll("LANDSAT/LE07/C02/T1_L2", prep_l57_ndbi)
    l8 = coll("LANDSAT/LC08/C02/T1_L2", prep_l89_ndbi)
    l9 = coll("LANDSAT/LC09/C02/T1_L2", prep_l89_ndbi)

    return ee.ImageCollection(
        ee.Algorithms.If(
            year.lte(2011), l5,
            ee.Algorithms.If(year.eq(2012), l7, l8.merge(l9))
        )
    )

print("NDBI functions defined.")

NDBI functions defined.


In [ ]:
def compute_year_ndbi(year):
    """Per-tract NDBI mean + impervious (built-up) fraction for one year."""
    year = ee.Number(year)
    coll = get_collection_ndbi(year).select(["NDBI", "builtup"])
    n_scenes = coll.size()

    composite = coll.median()             # median of 0/1 = per-pixel majority built

    result = composite.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=SCALE_M,
        tileScale=TILE_SCALE,
    )

    def tag(f):
        bp = ee.Number(f.get("builtup"))                  # fraction 0..1 (may be null)
        area_km2 = f.geometry().area(1).divide(1e6)
        return f.set({
            "year": year,
            "n_scenes": n_scenes,
            "imperv_percent": ee.Algorithms.If(bp, bp.multiply(100), None),
            "imperv_area_km2": ee.Algorithms.If(bp, bp.multiply(area_km2), None),
            "tract_area_km2": area_km2,
            "satellite": ee.Algorithms.If(
                year.lte(2011), "Landsat 5",
                ee.Algorithms.If(year.eq(2012), "Landsat 7", "Landsat 8/9")),
        })

    return result.map(tag)

print("compute_year_ndbi() defined.")

compute_year_ndbi() defined.


### 2.1 GHSL built-up product (5-year epochs)

In [ ]:
# --- GHSL config -----------------------------------------------------
# GHSL built-up SURFACE: band GHS_BUILT_S gives built-up surface area (m^2) per
# 100 m pixel. Dividing by pixel area (10000 m^2) yields a 0..1 built fraction.
GHSL_COLLECTION = "JRC/GHSL/P2023A/GHS_BUILT_S"   # multi-temporal, 5-year epochs
GHSL_PIXEL_AREA_M2 = 100 * 100                    # 100 m pixels -> 10,000 m^2

# Census-aligned epochs to compute (GHSL provides 5-year steps).
# Adjust to match the census years you actually use.
GHSL_EPOCHS = [2000, 2005, 2010, 2015, 2020]      # add 2025 if present in your release

EXPORT_NAME_GHSL = "ville_montreal_ct_ghsl_builtup_epochs"

print("GHSL config loaded. Epochs:", GHSL_EPOCHS)

GHSL config loaded. Epochs: [2000, 2005, 2010, 2015, 2020]


In [ ]:
def compute_epoch_ghsl(epoch):
    """Per-tract GHSL built-up FRACTION (0..1) for a 5-year epoch.

    GHS_BUILT_S stores built-up surface in m^2 per 100 m pixel. The pixel built
    fraction is built_m2 / 10,000; the per-tract mean is the built share of area.
    Images are addressed directly by ID with the year as a path suffix, e.g.
    'JRC/GHSL/P2023A/GHS_BUILT_S/2020' (the documented access pattern).
    """
    # epoch is a plain Python int here (mapped client-side over GHSL_EPOCHS)
    img = ee.Image("%s/%d" % (GHSL_COLLECTION, epoch))

    built_frac = (img.select("built_surface")
                  .divide(GHSL_PIXEL_AREA_M2)
                  .clamp(0, 1)
                  .rename("ghsl_built"))


    result = built_frac.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=100,                 # GHSL native resolution
        tileScale=TILE_SCALE,
    )

    def tag(f):
        b = ee.Number(f.get("mean"))
        return f.set({
            "year": epoch,
            "ghsl_built": b,
            "ghsl_built_percent": ee.Algorithms.If(b, b.multiply(100), None),
            "source": "GHSL GHS_BUILT_S",
        })
    return result.map(tag)

print("compute_epoch_ghsl() defined (GHSL built-up fraction, image-by-ID, 5-year epochs).")

compute_epoch_ghsl() defined (GHSL built-up fraction, image-by-ID, 5-year epochs).


### 2.2 Calibrate NDBI threshold against GHSL (auto-set)

In [ ]:
# === NDBI threshold calibration against GHSL (function) ===================
# GHSL (built-up SURFACE product) is the calibration reference — it is conceptually
# closer to "built-up fraction" than Dynamic World (a land-cover classifier that can
# overestimate built area). We pick the NDBI threshold whose city-wide built fraction
# best matches GHSL in the calibration year. NO water mask is used anywhere.
def calibrate_ndbi_threshold_ghsl(calib_year=None, sweep=None, verbose=True):
    """Find the NDBI threshold whose city-wide Landsat built fraction best matches
    the GHSL built fraction in the nearest GHSL epoch. Returns the threshold and
    prints the bias (magnitude + direction) BEFORE and AFTER calibration."""
    calib_year = calib_year or 2015
    sweep = sweep or [-0.30, -0.25, -0.20, -0.15, -0.10, -0.05, 0.00, 0.05, 0.10]

    # GHSL reference built fraction (nearest epoch to calib_year)
    ghsl_epoch = min(GHSL_EPOCHS, key=lambda e: abs(e - calib_year))
    ghsl_fc = compute_epoch_ghsl(ghsl_epoch)
    ghsl_target = ghsl_fc.aggregate_mean("ghsl_built").getInfo()
    if ghsl_target is None:
        raise ValueError("GHSL target is None — check compute_epoch_ghsl and the "
                         "GHSL band name (run the GHSL sanity-check cell).")

    def ndbi_builtfrac_at(year, thr):
        comp = get_collection_ndbi(year).select("NDBI").median()
        built = comp.gt(thr).rename("b")        # NO water mask
        stat = built.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=tracts.geometry(),
            scale=SCALE_M, tileScale=TILE_SCALE, maxPixels=1e13)
        return stat.get("b").getInfo()

    if verbose:
        print(f"Calibration year: {calib_year}  (GHSL epoch used: {ghsl_epoch})")
        print(f"GHSL target built fraction: {round(ghsl_target,3)} "
              f"({round(ghsl_target*100,1)}%)\n")

    frac_before = ndbi_builtfrac_at(calib_year, 0.0)
    bias_before = (frac_before - ghsl_target) * 100
    if verbose:
        d = "underestimates" if bias_before < 0 else "overestimates"
        print("BIAS BEFORE calibration (NDBI threshold = 0.0):")
        print(f"  Landsat built %: {round(frac_before*100,1)} | "
              f"GHSL built %: {round(ghsl_target*100,1)}")
        print(f"  Landsat {d} GHSL by {abs(round(bias_before,1))} pp\n")
        print("Threshold sweep (|gap| to GHSL):")

    rows = []
    for thr in sweep:
        frac = ndbi_builtfrac_at(calib_year, thr)
        gap = None if frac is None else abs(frac - ghsl_target)
        rows.append((thr, frac, gap))
        if verbose:
            print(f"  NDBI>{thr:+.2f}: built %="
                  f"{None if frac is None else round(frac*100,1)} | "
                  f"|gap|={None if gap is None else round(gap*100,1)} pp")

    valid = [(t,f,g) for (t,f,g) in rows if g is not None]
    if not valid:
        raise ValueError("All thresholds returned None — check Landsat collection.")
    thr_cal = min(valid, key=lambda r: r[2])[0]
    frac_after = {t:f for (t,f,g) in valid}[thr_cal]
    bias_after = (frac_after - ghsl_target) * 100

    if verbose:
        d = "underestimates" if bias_after < 0 else "overestimates"
        print(f"\nCalibrated NDBI threshold = {thr_cal}")
        print("BIAS AFTER calibration:")
        print(f"  Landsat built %: {round(frac_after*100,1)} | "
              f"GHSL built %: {round(ghsl_target*100,1)}")
        print(f"  Landsat {d} GHSL by {abs(round(bias_after,1))} pp")
        if abs(bias_after) <= 5:
            print("  -> Calibration ADEQUATE: residual bias within 5 pp.")
        else:
            print("  -> Calibration PARTIAL: residual bias > 5 pp; widen sweep or "
                  "use GHSL directly as the built-up variable.")

        # Cross-check the calibrated threshold against GHSL 2000 (back-validation)
        if 2000 in GHSL_EPOCHS:
            g2000 = compute_epoch_ghsl(2000).aggregate_mean("ghsl_built").getInfo()
            n2000 = ndbi_builtfrac_at(2000, thr_cal)
            if g2000 is not None and n2000 is not None:
                print(f"\nBack-validation @ 2000: NDBI built %={round(n2000*100,1)} "
                      f"vs GHSL built %={round(g2000*100,1)} "
                      f"(gap {round((n2000-g2000)*100,1)} pp)")
    return thr_cal

print("calibrate_ndbi_threshold_ghsl() defined.")

calibrate_ndbi_threshold_ghsl() defined.


In [ ]:
# Run GHSL calibration ONCE and set the global NDBI_THRESHOLD used by all built-up cells.
NDBI_THRESHOLD = calibrate_ndbi_threshold_ghsl(calib_year=2015)
print(f"\nNDBI_THRESHOLD is now {NDBI_THRESHOLD} for all subsequent cells.")

Calibration year: 2015  (GHSL epoch used: 2015)
GHSL target built fraction: 0.279 (27.9%)

BIAS BEFORE calibration (NDBI threshold = 0.0):
  Landsat built %: 17.1 | GHSL built %: 27.9
  Landsat underestimates GHSL by 10.9 pp

Threshold sweep (|gap| to GHSL):
  NDBI>-0.30: built %=81.5 | |gap|=53.6 pp
  NDBI>-0.25: built %=72.2 | |gap|=44.2 pp
  NDBI>-0.20: built %=61.4 | |gap|=33.4 pp
  NDBI>-0.15: built %=49.6 | |gap|=21.7 pp
  NDBI>-0.10: built %=37.9 | |gap|=9.9 pp
  NDBI>-0.05: built %=26.5 | |gap|=1.4 pp
  NDBI>+0.00: built %=17.1 | |gap|=10.9 pp
  NDBI>+0.05: built %=8.9 | |gap|=19.1 pp
  NDBI>+0.10: built %=3.9 | |gap|=24.0 pp

Calibrated NDBI threshold = -0.05
BIAS AFTER calibration:
  Landsat built %: 26.5 | GHSL built %: 27.9
  Landsat underestimates GHSL by 1.4 pp
  -> Calibration ADEQUATE: residual bias within 5 pp.

Back-validation @ 2000: NDBI built %=20.3 vs GHSL built %=27.4 (gap -7.1 pp)

NDBI_THRESHOLD is now -0.05 for all subsequent cells.


### 2.3 Validation — NDBI range, bounds, and threshold sensitivity

In [ ]:
if NDBI_THRESHOLD is None:
    raise ValueError("Run the calibration cell first (sets NDBI_THRESHOLD).")

# NDBI range + impervious fraction bounds for one representative year
gb = ee.FeatureCollection(ee.List([PNG_YEAR]).map(compute_year_ndbi)).flatten()

ndbi_stats = gb.reduceColumns(ee.Reducer.minMax(), ["NDBI"]).getInfo()
imp_stats  = gb.reduceColumns(ee.Reducer.minMax(), ["builtup"]).getInfo()
print(f"{PNG_YEAR} NDBI    min/max (expect within [-1,1]):", ndbi_stats)
print(f"{PNG_YEAR} builtup min/max (expect 0..1):", imp_stats)

mn, mx = imp_stats.get("min"), imp_stats.get("max")
assert (mn is None) or (mn >= -1e-9 and mx <= 1 + 1e-9), \
    "builtup out of [0,1] — check masking/threshold."
print("Impervious fraction range OK.")

2015 NDBI    min/max (expect within [-1,1]): {'max': 0.08734741917990684, 'min': -0.3583770297079616}
2015 builtup min/max (expect 0..1): {'max': 0.920130659687159, 'min': 0.002287078200038757}
Impervious fraction range OK.


In [ ]:
# Threshold sensitivity for NDBI: rerun one year at -0.05 / 0.0 / 0.05
def builtup_at(y, thr):
    comp = get_collection_ndbi(y).select("NDBI").median()
    built = comp.gt(thr).rename("b")
    stat = built.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=tracts.geometry(),
        scale=SCALE_M,
        tileScale=TILE_SCALE,
        maxPixels=1e13,
    )
    return stat.get("b").getInfo()        # band named "b" -> key exists

print(f"NDBI threshold sensitivity for {PNG_YEAR}:")
for thr in (-0.05, 0.0, 0.05):
    val = builtup_at(PNG_YEAR, thr)
    print(f"  NDBI>{thr:+.2f}: mean built-up = "
          f"{round(val, 3) if val is not None else None}")

NDBI threshold sensitivity for 2015:
  NDBI>-0.05: mean built-up = 0.265
  NDBI>+0.00: mean built-up = 0.171
  NDBI>+0.05: mean built-up = 0.089


NDBI Threshold Calibration (GHSL Reference, 2015)

GHSL built-up fraction (2015): 27.9%

Tested thresholds:

NDBI > -0.05 → 26.5%
NDBI > 0.00 → 17.1%
NDBI > 0.05 → 8.9%

The threshold of -0.05 best matches the GHSL estimate (difference = 1.4 percentage points) and is therefore used for all subsequent NDBI built-up calculations.

Selected threshold:

NDBI_THRESHOLD = -0.05

GHSL was selected as the calibration reference because it directly measures built-up surface area, whereas Dynamic World is a land-cover classification product that may classify mixed urban areas differently. The calibrated threshold allows slightly negative NDBI values to be classified as built-up, capturing residential and mixed urban areas that would otherwise be omitted when using the conventional threshold of 0.0.

### 2.4 Exports — impervious series

In [ ]:
def export_block_ndbi(year_list, name):
    fc = ee.FeatureCollection(ee.List(year_list).map(compute_year_ndbi)).flatten()
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=name,
        folder=DRIVE_OUTPUT_FOLDER,
        fileFormat="CSV",
        selectors=["year", "satellite", "n_scenes", "CTUID", "CTNAME", "DGUID",
                   "NDBI", "builtup", "imperv_percent",
                   "imperv_area_km2", "tract_area_km2"],
    )
    task.start()
    print(f"Export started -> {name} (Drive/{DRIVE_OUTPUT_FOLDER})")
    return task

ti1 = export_block_ndbi(list(range(2000, 2012)), EXPORT_NAMES_IMPERV["p1"])  # L5
ti2 = export_block_ndbi([2012],                  EXPORT_NAMES_IMPERV["p2"])  # L7
ti3 = export_block_ndbi(list(range(2013, END_YEAR + 1)), EXPORT_NAMES_IMPERV["p3"])  # L8/9
print("Impervious CSV export tasks submitted.")

Export started -> ville_montreal_ct_ndbi_imperv_2000_2011 (Drive/IRIU_VF)
Export started -> ville_montreal_ct_ndbi_imperv_2012 (Drive/IRIU_VF)
Export started -> ville_montreal_ct_ndbi_imperv_2013_2025 (Drive/IRIU_VF)
Impervious CSV export tasks submitted.


### 2.5 GHSL sanity check & export

In [ ]:
fc_ghsl = ee.FeatureCollection([compute_epoch_ghsl(e) for e in GHSL_EPOCHS]).flatten()
task_ghsl = ee.batch.Export.table.toDrive(
    collection=fc_ghsl,
    description=EXPORT_NAME_GHSL,
    folder=DRIVE_OUTPUT_FOLDER,
    fileFormat="CSV",
    selectors=["year", "source", "CTUID", "CTNAME", "DGUID",
               "ghsl_built", "ghsl_built_percent"],
)
task_ghsl.start()
print(f"GHSL export started -> {EXPORT_NAME_GHSL} (Drive/{DRIVE_OUTPUT_FOLDER})")

GHSL export started -> ville_montreal_ct_ghsl_builtup_epochs (Drive/IRIU_VF)


### 2.6 Three-way comparison per tract (choose your variable)

In [ ]:
import pandas as pd

COMPARE_YEAR = 2015
GHSL_EPOCH = 2015
_thr = NDBI_THRESHOLD

def _fc_to_df(fc, cols):
    feats = fc.select(cols, retainGeometry=False).getInfo()["features"]
    return pd.DataFrame([f["properties"] for f in feats])

def compute_year_ndbi_at(year, thr):
    comp = get_collection_ndbi(year).select("NDBI").median()
    built = comp.gt(thr).rename("ndbi_built")

    r = built.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=SCALE_M,
        tileScale=TILE_SCALE
    )

    return r.map(lambda f: f.set({
        "CTUID": f.get("CTUID"),
        "ndbi_built": f.get("mean")
    }))

ndbi_fc = compute_year_ndbi_at(COMPARE_YEAR, _thr)
ghsl_fc = compute_epoch_ghsl(GHSL_EPOCH)

df_ndbi = _fc_to_df(ndbi_fc, ["CTUID", "ndbi_built"])
df_ghsl = _fc_to_df(ghsl_fc, ["CTUID", "ghsl_built"])

df = df_ndbi.merge(df_ghsl, on="CTUID", how="outer")

print(f"Comparison year: {COMPARE_YEAR} | GHSL epoch: {GHSL_EPOCH}")
print(f"NDBI threshold: {_thr}")

print("\nSummary statistics:")
print(df[["ndbi_built", "ghsl_built"]].describe().round(3))

print("\nCorrelation:")
print(df[["ndbi_built", "ghsl_built"]].corr().round(3))

print("\nMean difference:")
print((df["ndbi_built"] - df["ghsl_built"]).mean().round(3))

df.to_csv("ndbi_vs_ghsl_comparison_2015.csv", index=False)
print("\nSaved: ndbi_vs_ghsl_comparison_2015.csv")

Comparison year: 2015 | GHSL epoch: 2015
NDBI threshold: -0.05

Summary statistics:
       ndbi_built  ghsl_built
count     484.000     484.000
mean        0.269       0.279
std         0.192       0.083
min         0.002       0.014
25%         0.106       0.243
50%         0.232       0.299
75%         0.393       0.336
max         0.919       0.439

Correlation:
            ndbi_built  ghsl_built
ndbi_built       1.000       0.199
ghsl_built       0.199       1.000

Mean difference:
-0.01

Saved: ndbi_vs_ghsl_comparison_2015.csv


Validation summary (2015)

After calibration against GHSL, the NDBI threshold was set to -0.05. The city-wide built-up fraction estimated by NDBI (26.9%) closely matches the GHSL reference (27.9%), with a difference of only ~1 percentage point.

However, the tract-level correlation between NDBI and GHSL is low (r = 0.199), indicating that NDBI reproduces the overall amount of built-up area reasonably well but does not closely match the spatial distribution observed in GHSL.

For modeling:

GHSL should be preferred when available (2000, 2005, 2010, 2015, and 2020).
Calibrated NDBI can be used as an annual proxy between GHSL epochs.
Final variable selection should be based on model performance and validation results.

---
## 3. Land Surface Temperature (LST)

Surface temperature is the most directly heat-relevant exposure variable. It is
computed from the **Landsat Collection 2 Level-2 surface-temperature band** (already
atmospherically corrected and emissivity-adjusted by USGS), so no manual thermal
processing is needed.

**Four variables per tract-year**, so the model can test which works best for heatwaves:

| Variable | Definition |
|---|---|
| `lst_median_c` | summer **median** LST (\u00b0C) \u2014 typical summer surface temperature |
| `lst_max_c` | summer **maximum** LST (\u00b0C) \u2014 extreme surface temperature |
| `lst_median_rel` | median LST minus the city-wide mean median that year (surface UHI) |
| `lst_max_rel` | maximum LST minus the city-wide mean maximum that year |

**Relative LST** (tract minus city mean) is recommended for cross-year and
cross-sensor comparison: it absorbs differences in overpass time, summer warmth, and
the 2012/2013 sensor change, isolating which neighbourhoods are hotter than the city.

**Thermal scaling (Collection 2):** `LST_K = ST_B* x 0.00341802 + 149.0`, then
`LST_C = LST_K - 273.15`. Bands: `ST_B6` (Landsat 5/7), `ST_B10` (Landsat 8/9).

> Reuses `mask_landsat_sr`, `SUMMER_START/END`, `START_YEAR/END_YEAR`, `SCALE_M`,
> `TILE_SCALE`, `tracts`, `DRIVE_OUTPUT_FOLDER`, `PNG_YEAR`.

In [ ]:
# --- LST config (edit here) ------------------------------------------
# Collection-2 surface-temperature scale/offset (do NOT use the optical 0.0000275/-0.2)
ST_SCALE, ST_OFFSET = 0.00341802, 149.0     # -> Kelvin
KELVIN_TO_C = -273.15

# Thermal band per sensor family (Collection 2 L2)
ST_BAND_L57 = "ST_B6"     # Landsat 5 TM / 7 ETM+
ST_BAND_L89 = "ST_B10"    # Landsat 8 / 9 OLI-TIRS

# Plausible-range filter for cloud/edge artefacts on LST (deg C)
LST_MIN_C, LST_MAX_C = -20.0, 65.0

EXPORT_NAMES_LST = {
    "p1": "ville_montreal_ct_lst_2000_2011",
    "p2": "ville_montreal_ct_lst_2012",
    "p3": "ville_montreal_ct_lst_2013_2025",
}
print("LST config loaded | bands L5/7=ST_B6, L8/9=ST_B10 | scale/offset for Kelvin")

LST config loaded | bands L5/7=ST_B6, L8/9=ST_B10 | scale/offset for Kelvin


In [ ]:
# --- Per-image LST in Celsius, cloud-masked --------------------------
def _add_lst_c(image, st_band):
    """Scale Collection-2 ST band to Celsius and add as 'LST_C'. Cloud/shadow/snow
    are masked via the shared mask_landsat_sr (essential for thermal: clouds are cold
    and would bias LST downward)."""
    image = mask_landsat_sr(image)                       # reuse QA mask
    lst_c = (image.select(st_band)
             .multiply(ST_SCALE).add(ST_OFFSET)          # -> Kelvin
             .add(KELVIN_TO_C)                           # -> Celsius
             .rename("LST_C"))
    # drop physically implausible values (residual cloud / sensor edge)
    lst_c = lst_c.updateMask(lst_c.gte(LST_MIN_C).And(lst_c.lte(LST_MAX_C)))
    return image.addBands(lst_c)

def prep_l57_lst(image):
    return _add_lst_c(image, ST_BAND_L57)

def prep_l89_lst(image):
    return _add_lst_c(image, ST_BAND_L89)

def get_collection_lst(year):
    """Masked, LST-tagged Landsat collection for one summer, sensor-selected."""
    year  = ee.Number(year)
    start = ee.Date.fromYMD(year, SUMMER_START[0], SUMMER_START[1])
    end   = ee.Date.fromYMD(year, SUMMER_END[0],   SUMMER_END[1])
    def coll(cid, prep):
        return (ee.ImageCollection(cid).filterDate(start, end)
                .filterBounds(tracts).map(prep))
    l5 = coll("LANDSAT/LT05/C02/T1_L2", prep_l57_lst)
    l7 = coll("LANDSAT/LE07/C02/T1_L2", prep_l57_lst)
    l8 = coll("LANDSAT/LC08/C02/T1_L2", prep_l89_lst)
    l9 = coll("LANDSAT/LC09/C02/T1_L2", prep_l89_lst)
    return ee.ImageCollection(
        ee.Algorithms.If(year.lte(2011), l5,
            ee.Algorithms.If(year.eq(2012), l7, l8.merge(l9))))

print("LST band prep + get_collection_lst() defined.")

LST band prep + get_collection_lst() defined.


In [ ]:
def compute_year_lst(year):
    """Per-tract summer median & maximum LST (absolute deg C + relative to city mean)."""
    year = ee.Number(year)
    coll = get_collection_lst(year).select("LST_C")
    n_scenes = coll.size()

    lst_median = coll.median().rename("lst_median_c")
    lst_max    = coll.max().rename("lst_max_c")
    composite  = lst_median.addBands(lst_max)

    # Per-tract absolute values
    result = composite.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=SCALE_M,
        tileScale=TILE_SCALE,
    )

    # City-wide means this year (for the relative / surface-UHI version)
    city = composite.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=tracts.geometry(),
        scale=SCALE_M, tileScale=TILE_SCALE, maxPixels=1e13)
    city_med = ee.Number(city.get("lst_median_c"))
    city_max = ee.Number(city.get("lst_max_c"))

    def tag(f):
        med = ee.Number(f.get("lst_median_c"))
        mx  = ee.Number(f.get("lst_max_c"))
        return f.set({
            "year": year,
            "n_scenes": n_scenes,
            "lst_median_c": med,
            "lst_max_c": mx,
            "lst_median_rel": ee.Algorithms.If(med, med.subtract(city_med), None),
            "lst_max_rel":    ee.Algorithms.If(mx,  mx.subtract(city_max),  None),
            "satellite": ee.Algorithms.If(
                year.lte(2011), "Landsat 5",
                ee.Algorithms.If(year.eq(2012), "Landsat 7", "Landsat 8/9")),
        })
    return result.map(tag)

print("compute_year_lst() defined.")

compute_year_lst() defined.


### 3.1 Validation

In [ ]:
# Validate LST for a representative year: ranges + scene count.
lst_fc = compute_year_lst(2015)

med_stats = lst_fc.reduceColumns(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    selectors=["lst_median_c"]).getInfo()
max_stats = lst_fc.reduceColumns(ee.Reducer.minMax(), ["lst_max_c"]).getInfo()
rel_stats = lst_fc.reduceColumns(ee.Reducer.minMax(), ["lst_median_rel"]).getInfo()
n = lst_fc.aggregate_first("n_scenes").getInfo()

print(f"Year 2015 (scenes={n})")
print("  median LST (C) min/mean/max:", med_stats)
print("  max    LST (C) min/max     :", max_stats)
print("  relative median LST (C) min/max:", rel_stats, "(centered near 0)")

mn = med_stats.get("lst_median_c_min"); mx = med_stats.get("lst_median_c_max")
assert (mn is None) or (mn >= LST_MIN_C and mx <= LST_MAX_C), \
    "median LST outside plausible range — check scaling/masking."
print("LST ranges OK. Max should exceed median; relative values straddle 0.")
if n is not None and n < 4:
    print(f" WARNING: only {n} scenes for 2015 - thermal composite may be noisy.")

Year 2015 (scenes=21)
  median LST (C) min/mean/max: {'max': 36.913672688064054, 'mean': 33.0164656714199, 'min': 23.14936720766268}
  max    LST (C) min/max     : {'max': 40.98382289330367, 'min': 25.00813527281359}
  relative median LST (C) min/max: {'max': 5.905353405736296, 'min': -7.858952074665076} (centered near 0)
LST ranges OK. Max should exceed median; relative values straddle 0.


### 3.2 Export (2000–2025, three sensor blocks)

In [ ]:
def export_block_lst(year_list, name):
    fc = ee.FeatureCollection(ee.List(year_list).map(compute_year_lst)).flatten()
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=name,
        folder=DRIVE_OUTPUT_FOLDER,
        fileFormat="CSV",
        selectors=["year", "satellite", "n_scenes", "CTUID", "CTNAME", "DGUID",
                   "lst_median_c", "lst_max_c", "lst_median_rel", "lst_max_rel"],
    )
    task.start()
    print(f"LST export started -> {name} (Drive/{DRIVE_OUTPUT_FOLDER})")
    return task

tl1 = export_block_lst(list(range(START_YEAR, 2012)), EXPORT_NAMES_LST["p1"])  # L5
tl2 = export_block_lst([2012],                        EXPORT_NAMES_LST["p2"])  # L7
tl3 = export_block_lst(list(range(2013, END_YEAR + 1)), EXPORT_NAMES_LST["p3"])  # L8/9
print("All LST export tasks submitted.")

LST export started -> ville_montreal_ct_lst_2000_2011 (Drive/IRIU_VF)
LST export started -> ville_montreal_ct_lst_2012 (Drive/IRIU_VF)
LST export started -> ville_montreal_ct_lst_2013_2025 (Drive/IRIU_VF)
All LST export tasks submitted.


**Modelling notes for heatwaves.**
- `lst_max_c` / `lst_max_rel` target extreme surface heat; `lst_median_*` give the
  typical-summer baseline. Test both; do not enter median and max together raw
  (they are strongly correlated).
- Prefer the **relative** versions for cross-year modelling \u2014 they absorb overpass-time,
  inter-annual warmth, and the 2012/2013 sensor change, isolating the neighbourhood
  surface-UHI signal.
- Flag **2012** (Landsat-7 SLC-off gaps) and any low-`n_scenes` year as lower-confidence.
- LST is the near-inverse of green cover and correlates with built-up; check
  collinearity before combining them in a model.

---
## 4. Tree Canopy

Tree canopy is the vegetation component most relevant to heat: trees provide shade
and evapotranspiration that grass and shrubs do not. Isolating *canopy* (trees)
from *all vegetation* (which green cover already captures) is difficult at Landsat's
30 m resolution, and the dedicated high-quality canopy products do not reach back to
2000. We therefore follow the **same calibrated-proxy strategy used for built-up**:

- **Annual canopy proxy (2000\u20132025):** a *higher* NDVI threshold than the green-cover
  cutoff approximates dense woody vegetation. Trees hold high summer NDVI more
  consistently than grass (which browns), so a high-NDVI mask is a reasonable canopy
  proxy. Built from the same Landsat pipeline, so it is annual and internally consistent.
- **Calibration reference (recent years):** Dynamic World's **'trees' class** (2015+),
  a genuine tree classification. The proxy threshold is chosen so the proxy's canopy %
  matches Dynamic World trees % in an overlap year \u2014 exactly as NDBI was calibrated to GHSL.

| Variable | Definition |
|---|---|
| `canopy_percent` | % of tract pixels above the (calibrated) canopy NDVI threshold |
| `canopy_area_km2` | canopy fraction \u00d7 tract area |

> **Caveat (important):** even calibrated, a 30 m canopy proxy is the **weakest** of the
> environmental variables \u2014 tree/grass separation is intrinsically hard at this
> resolution. It is reasonable city-wide and rough at the tract level; interpret
> tract values as indicative. If canopy is central (not supporting) to the model,
> prefer the Dynamic World trees layer as primary for 2015+ and use the proxy only to
> extend backward.

> Reuses `get_collection` (green-cover NDVI collection), `tracts`, `SCALE_M`,
> `TILE_SCALE`, `SUMMER_*`, `START_YEAR/END_YEAR`, `DRIVE_OUTPUT_FOLDER`, `PNG_YEAR`.

In [ ]:
# --- Tree canopy config (edit here) ----------------------------------
# Canopy NDVI cutoff is HIGHER than the green-cover cutoff (dense woody vegetation).
# SET BY CALIBRATION below against Dynamic World 'trees'; placeholder until then.
CANOPY_NDVI_THRESHOLD = None      # auto-set by the calibration step (Section 16.1)

DW_START_YEAR = 2021
DW_TREES_CLASS = 1                # Dynamic World label value for 'trees'
CANOPY_CALIB_YEAR = max(PNG_YEAR, DW_START_YEAR)   # overlap year (>=2015)

EXPORT_NAMES_CANOPY = {
    "p1": "ville_montreal_ct_canopy_2000_2011",
    "p2": "ville_montreal_ct_canopy_2012",
    "p3": "ville_montreal_ct_canopy_2013_2025",
}
print("Canopy config loaded | threshold set by calibration vs Dynamic World 'trees'")

Canopy config loaded | threshold set by calibration vs Dynamic World 'trees'


In [ ]:
# --- Canopy proxy: high-NDVI fraction per tract ----------------------
# Reuses get_collection() (the green-cover NDVI collection). We re-threshold the
# median NDVI at the (higher) canopy cutoff, so canopy uses the SAME imagery and
# masking as green cover but a stricter vegetation cutoff.
def compute_year_canopy(year):
    """Per-tract canopy fraction (0..1) = share of pixels with median NDVI above
    the calibrated canopy threshold."""
    year = ee.Number(year)
    ndvi = get_collection(year).select("NDVI").median()
    canopy = ndvi.gt(CANOPY_NDVI_THRESHOLD).rename("canopy")
    n_scenes = get_collection(year).size()

    result = canopy.reduceRegions(
        collection=tracts, reducer=ee.Reducer.mean(),
        scale=SCALE_M, tileScale=TILE_SCALE)

    def tag(f):
        c = ee.Number(f.get("mean"))
        area_km2 = f.geometry().area(1).divide(1e6)
        return f.set({
            "year": year,
            "n_scenes": n_scenes,
            "canopy_percent": ee.Algorithms.If(c, c.multiply(100), None),
            "canopy_area_km2": ee.Algorithms.If(c, c.multiply(area_km2), None),
            "tract_area_km2": area_km2,
            "satellite": ee.Algorithms.If(
                year.lte(2011), "Landsat 5",
                ee.Algorithms.If(year.eq(2012), "Landsat 7", "Landsat 8/9")),
        })
    return result.map(tag)

print("compute_year_canopy() defined.")

compute_year_canopy() defined.


### 4.1 Calibrate the canopy NDVI threshold against Dynamic World trees

In [ ]:
# Sweep canopy NDVI thresholds; match proxy canopy % to Dynamic World 'trees' %.
def calibrate_canopy_threshold(calib_year=None, sweep=None, verbose=True):
    calib_year = calib_year or CANOPY_CALIB_YEAR
    sweep = sweep or [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

    dw_fc = compute_year_dw_trees(calib_year)
    dw_target = dw_fc.aggregate_mean("dw_trees").getInfo()
    if dw_target is None:
        raise ValueError("Dynamic World trees target is None — check compute_year_dw_trees.")

    def canopy_frac_at(year, thr):
        ndvi = get_collection(year).select("NDVI").median()
        c = ndvi.gt(thr).rename("mean")
        stat = c.reduceRegion(ee.Reducer.mean(), tracts.geometry(),
                              scale=SCALE_M, tileScale=TILE_SCALE, maxPixels=1e13)
        return stat.get("mean").getInfo()

    if verbose:
        print(f"Calibration year: {calib_year}")
        print(f"Dynamic World 'trees' target: {round(dw_target,3)} "
              f"({round(dw_target*100,1)}%)\n")
        print("Threshold sweep (|gap| to DW trees):")
    rows=[]
    for thr in sweep:
        frac = canopy_frac_at(calib_year, thr)
        gap = None if frac is None else abs(frac - dw_target)
        rows.append((thr, frac, gap))
        if verbose:
            print(f"  NDVI>{thr:.2f}: canopy %={None if frac is None else round(frac*100,1)} "
                  f"| |gap|={None if gap is None else round(gap*100,1)} pp")
    valid=[(t,f,g) for (t,f,g) in rows if g is not None]
    if not valid: raise ValueError("All thresholds None — check inputs.")
    thr_cal = min(valid, key=lambda r:r[2])[0]
    frac_after = {t:f for (t,f,g) in valid}[thr_cal]
    if verbose:
        d = "underestimates" if (frac_after-dw_target)<0 else "overestimates"
        print(f"\nCalibrated canopy NDVI threshold = {thr_cal}")
        print(f"Proxy {d} DW trees by {abs(round((frac_after-dw_target)*100,1))} pp after calibration.")
        if abs(frac_after-dw_target) <= 5/100:
            print("  -> ADEQUATE (within 5 pp).")
        else:
            print("  -> PARTIAL (>5 pp); canopy proxy is approximate at 30 m. "
                  "Consider DW trees as primary for 2015+.")
    return thr_cal

CANOPY_NDVI_THRESHOLD = calibrate_canopy_threshold()
print(f"\nCANOPY_NDVI_THRESHOLD is now {CANOPY_NDVI_THRESHOLD} for all subsequent cells.")

Calibration year: 2021
Dynamic World 'trees' target: 0.025 (2.5%)

Threshold sweep (|gap| to DW trees):
  NDVI>0.35: canopy %=48.5 | |gap|=46.0 pp
  NDVI>0.40: canopy %=41.7 | |gap|=39.1 pp
  NDVI>0.45: canopy %=34.9 | |gap|=32.4 pp
  NDVI>0.50: canopy %=28.7 | |gap|=26.2 pp
  NDVI>0.55: canopy %=22.9 | |gap|=20.3 pp
  NDVI>0.60: canopy %=17.7 | |gap|=15.1 pp
  NDVI>0.65: canopy %=13.4 | |gap|=10.8 pp
  NDVI>0.70: canopy %=10.0 | |gap|=7.5 pp

Calibrated canopy NDVI threshold = 0.7
Proxy overestimates DW trees by 7.5 pp after calibration.
  -> PARTIAL (>5 pp); canopy proxy is approximate at 30 m. Consider DW trees as primary for 2015+.

CANOPY_NDVI_THRESHOLD is now 0.7 for all subsequent cells.


### 4.2 Validation

In [ ]:
if CANOPY_NDVI_THRESHOLD is None:
    raise ValueError("Run the Section 16.1 calibration cell first.")

cfc = compute_year_canopy(PNG_YEAR)
stats = cfc.reduceColumns(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    selectors=["canopy_percent"]).getInfo()
n = cfc.aggregate_first("n_scenes").getInfo()
print(f"Year {PNG_YEAR} canopy % min/mean/max:", stats, f"| scenes={n}")

# canopy % should be <= green-cover % (canopy is a subset of all vegetation)
print("Sanity: canopy % should not exceed green-cover % for the same tracts.")
print("(Compare against your Section-11 green-cover output for PNG_YEAR.)")

Year 2015 canopy % min/mean/max: {'max': 85.48412484259755, 'mean': 8.584735453345456, 'min': 0.0007594859798888111} | scenes=21
Sanity: canopy % should not exceed green-cover % for the same tracts.
(Compare against your Section-11 green-cover output for PNG_YEAR.)


### Interpretation of the Canopy Calibration

The NDVI-based canopy proxy was calibrated against the Dynamic World "trees" class using 2021 as the overlap year. The optimal NDVI threshold was 0.70; however, the resulting canopy estimate (10.0%) remained substantially higher than the Dynamic World tree estimate (2.5%), with a difference of 7.5 percentage points.

These results indicate that a high-NDVI threshold does not isolate tree canopy effectively at Landsat's 30 m resolution. Dense grass, shrubs, mixed vegetation, and tree cover may still be grouped together, causing the proxy to overestimate true canopy cover.

Therefore:

- **2015–2025:** Dynamic World "trees" should be used as the primary tree-canopy variable.
- **2000–2014:** The NDVI canopy proxy can be used as an approximate historical extension when no dedicated tree-canopy product is available.
- Canopy values derived from NDVI should be interpreted as indicative rather than exact measures of tree cover.

### 4.3 Export (2000–2025, three sensor blocks)

In [ ]:
def export_block_canopy(year_list, name):
    fc = ee.FeatureCollection(ee.List(year_list).map(compute_year_canopy)).flatten()
    task = ee.batch.Export.table.toDrive(
        collection=fc, description=name, folder=DRIVE_OUTPUT_FOLDER, fileFormat="CSV",
        selectors=["year","satellite","n_scenes","CTUID","CTNAME","DGUID",
                   "canopy_percent","canopy_area_km2","tract_area_km2"])
    task.start(); print(f"Canopy export started -> {name}"); return task

tc1 = export_block_canopy(list(range(START_YEAR,2012)), EXPORT_NAMES_CANOPY["p1"])
tc2 = export_block_canopy([2012],                       EXPORT_NAMES_CANOPY["p2"])
tc3 = export_block_canopy(list(range(2013,END_YEAR+1)), EXPORT_NAMES_CANOPY["p3"])
print("All canopy export tasks submitted.")

Canopy export started -> ville_montreal_ct_canopy_2000_2011
Canopy export started -> ville_montreal_ct_canopy_2012
Canopy export started -> ville_montreal_ct_canopy_2013_2025
All canopy export tasks submitted.


 **Canopy is a subset of green cover** (trees are a part of all vegetation), so the
  two are strongly correlated; do not enter both raw into a model \u2014 choose one,
  residualize, or use the canopy/green ratio.
- **Treat as the weakest layer.** The 30 m proxy cannot cleanly separate trees from
  dense grass; tract values are indicative. For 2015+, the Dynamic World 'trees'
  fraction (`compute_year_dw_trees`) is a more reliable alternative.
- Flag **2012** (Landsat-7 SLC-off) and low-`n_scenes` years as lower-confidence.

"""
## 4.4 Tree Canopy Cover Using Dynamic World

In this section, we compute annual tree canopy cover at the census tract level using the Dynamic World land-cover dataset. Dynamic World is a 10 m resolution global land-cover product derived from Sentinel-2 imagery and provides classifications for several land-cover classes, including trees.

For each year between 2015 and 2025, summer observations (June–September) are aggregated and the dominant land-cover class is identified for each pixel. Pixels classified as trees are then used to estimate the percentage of each census tract covered by tree canopy.

Dynamic World was selected as the primary source for canopy estimation because it directly identifies tree cover, whereas NDVI-based approaches only provide an indirect measure of vegetation greenness and may not effectively distinguish trees from grass, shrubs, or other vegetation types. A calibration exercise comparing an NDVI canopy proxy with Dynamic World tree cover showed that the NDVI approach consistently overestimated canopy coverage, even when using high NDVI thresholds.

Therefore, Dynamic World tree canopy should be considered the preferred canopy variable for the period 2015–2025. NDVI-derived canopy estimates may still be useful as an approximate historical extension for years prior to 2015 when Dynamic World data are not available, but they should be interpreted with caution.
"""

In [ ]:
# ============================================================
# Dynamic World Trees (% canopy) by Census Tract
# Years: 2015-2025
# ============================================================

DW_TREES_CLASS = 1   # Dynamic World label for Trees

def compute_year_dw_trees(year):

    start = ee.Date.fromYMD(year, SUMMER_START[0], SUMMER_START[1])
    end   = ee.Date.fromYMD(year, SUMMER_END[0], SUMMER_END[1])

    dw = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(start, end)
        .filterBounds(tracts)
    )

    n_scenes = dw.size()

    # Most common summer class
    label = dw.select("label").mode()

    trees = label.eq(DW_TREES_CLASS).unmask(0).rename("trees")

    result = trees.reduceRegions(
        collection=tracts,
        reducer=ee.Reducer.mean(),
        scale=10,
        tileScale=TILE_SCALE
    )

    def tag(f):
        t = ee.Number(f.get("mean"))
        area_km2 = f.geometry().area(1).divide(1e6)

        return f.set({
            "year": year,
            "n_scenes": n_scenes,
            "tree_pct": ee.Algorithms.If(t, t.multiply(100), None),
            "tree_area_km2": ee.Algorithms.If(t, t.multiply(area_km2), None),
            "tract_area_km2": area_km2
        })

    return result.map(tag)


# ------------------------------------------------------------
# Build all years
# ------------------------------------------------------------

years = list(range(2015, 2026))

tree_fc = ee.FeatureCollection([])

for y in years:
    print("Processing", y)
    tree_fc = tree_fc.merge(compute_year_dw_trees(y))

print("Features:", tree_fc.size().getInfo())

Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
Processing 2025
Features: 5324


In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=tree_fc,
    description="montreal_ct_dynamicworld_trees_2015_2025",
    folder=DRIVE_OUTPUT_FOLDER,
    fileNamePrefix="montreal_ct_dynamicworld_trees_2015_2025",
    fileFormat="CSV",
    selectors=[
        "CTUID",
        "DGUID",
        "year",
        "tree_pct",
        "tree_area_km2",
        "tract_area_km2",
        "n_scenes"
    ]
)

task.start()

print("Export started.")

Export started.
